# LoRA Fine-Tuning on RoBERTa
Fine-tunes RoBERTa on MRPC, CoLA, and STS-B from GLUE using both full fine-tuning and LoRA (Low-Rank Adaptation), then compares results.

In [ ]:
!pip install -q transformers datasets scipy scikit-learn

In [ ]:
import math
import copy
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from datasets import load_dataset
from scipy.stats import pearsonr, matthews_corrcoef
from sklearn.metrics import accuracy_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. LoRA Layer Implementation
Implements LoRA as described in the paper: for a weight matrix W, we add a low-rank update B·A where A ∈ R^(r×d) and B ∈ R^(d×r). During training only A and B are updated; W is frozen.

In [ ]:
class LoRALinear(nn.Module):
    """Wraps an existing nn.Linear with a low-rank LoRA update."""

    def __init__(self, linear: nn.Linear, r: int = 8, alpha: int = 16):
        super().__init__()
        self.linear = linear
        self.r = r
        self.scaling = alpha / r

        in_features = linear.in_features
        out_features = linear.out_features

        # A is initialized with Gaussian, B with zeros (so initial ΔW = 0)
        self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))

        # Freeze the original weight
        self.linear.weight.requires_grad_(False)
        if self.linear.bias is not None:
            self.linear.bias.requires_grad_(False)

    def forward(self, x):
        base_out = self.linear(x)
        # LoRA update: x @ A^T @ B^T  * scaling
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T
        return base_out + self.scaling * lora_out


def apply_lora_to_roberta(model: nn.Module, r: int = 8, alpha: int = 16):
    """Replace query and value projection linears in every attention layer with LoRALinear."""
    for layer in model.roberta.encoder.layer:
        attn = layer.attention.self
        attn.query = LoRALinear(attn.query, r=r, alpha=alpha)
        attn.value = LoRALinear(attn.value, r=r, alpha=alpha)
    return model

## 2. Classifier Head
A thin classification (or regression) head on top of RoBERTa's `[CLS]` token.

In [ ]:
class RobertaClassifier(nn.Module):
    def __init__(self, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        hidden = self.roberta.config.hidden_size  # 768
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        return self.classifier(self.dropout(cls))

## 3. Dataset Preparation

In [ ]:
MAX_LEN = 128
BATCH_SIZE = 32

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')


def tokenize_pair(batch, key1, key2):
    return tokenizer(
        batch[key1], batch[key2],
        truncation=True, padding='max_length', max_length=MAX_LEN
    )

def tokenize_single(batch, key):
    return tokenizer(
        batch[key],
        truncation=True, padding='max_length', max_length=MAX_LEN
    )


def get_dataloader(dataset, label_col, batch_size=BATCH_SIZE, shuffle=True):
    dataset = dataset.with_format('torch')
    def collate(batch):
        input_ids = torch.stack([b['input_ids'] for b in batch])
        attention_mask = torch.stack([b['attention_mask'] for b in batch])
        labels = torch.tensor([b[label_col] for b in batch])
        return input_ids, attention_mask, labels
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)


# --- MRPC (sentence pair, binary classification) ---
mrpc_raw = load_dataset('glue', 'mrpc')
mrpc = mrpc_raw.map(lambda b: tokenize_pair(b, 'sentence1', 'sentence2'), batched=True)
mrpc = mrpc.remove_columns(['sentence1', 'sentence2', 'idx'])
mrpc = mrpc.rename_column('label', 'labels')

mrpc_train = get_dataloader(mrpc['train'], 'labels')
mrpc_val   = get_dataloader(mrpc['validation'], 'labels', shuffle=False)

# --- CoLA (single sentence, binary classification) ---
cola_raw = load_dataset('glue', 'cola')
cola = cola_raw.map(lambda b: tokenize_single(b, 'sentence'), batched=True)
cola = cola.remove_columns(['sentence', 'idx'])
cola = cola.rename_column('label', 'labels')

cola_train = get_dataloader(cola['train'], 'labels')
cola_val   = get_dataloader(cola['validation'], 'labels', shuffle=False)

# --- STS-B (sentence pair, regression 0-5 → normalize to 0-1) ---
stsb_raw = load_dataset('glue', 'stsb')

def normalize_stsb(batch):
    batch['label'] = [s / 5.0 for s in batch['label']]
    return batch

stsb = stsb_raw.map(normalize_stsb, batched=True)
stsb = stsb.map(lambda b: tokenize_pair(b, 'sentence1', 'sentence2'), batched=True)
stsb = stsb.remove_columns(['sentence1', 'sentence2', 'idx'])
stsb = stsb.rename_column('label', 'labels')

stsb_train = get_dataloader(stsb['train'], 'labels')
stsb_val   = get_dataloader(stsb['validation'], 'labels', shuffle=False)

print('Datasets loaded.')

## 4. Training and Evaluation Utilities

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, task):
    model.train()
    total_loss = 0.0
    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)

        if task == 'regression':
            loss = loss_fn(logits.squeeze(-1), labels.float())
        else:
            loss = loss_fn(logits, labels.long())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, task):
    model.eval()
    all_preds, all_labels = [], []
    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        logits = model(input_ids, attention_mask)

        if task == 'regression':
            preds = logits.squeeze(-1).cpu().numpy()
        else:
            preds = logits.argmax(dim=-1).cpu().numpy()

        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

    if task == 'accuracy':
        return accuracy_score(all_labels, all_preds)
    elif task == 'matthews':
        return matthews_corrcoef(all_labels, all_preds)
    elif task == 'regression':
        r, _ = pearsonr(all_labels, all_preds)
        return r


def run_training(model, train_loader, val_loader, task, epochs=3, lr=2e-4):
    if task == 'regression':
        loss_fn = nn.MSELoss()
    else:
        loss_fn = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=0.01
    )

    best_metric = -float('inf')
    for epoch in range(1, epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, loss_fn, task)
        metric = evaluate(model, val_loader, task)
        print(f'  Epoch {epoch}/{epochs}  loss={loss:.4f}  val={metric:.4f}')
        if metric > best_metric:
            best_metric = metric

    return best_metric

## 5. Full Fine-Tuning Baseline

In [ ]:
results = {}

# ---- MRPC full fine-tune ----
print('=== Full Fine-Tune: MRPC ===')
model_full_mrpc = RobertaClassifier(num_labels=2).to(DEVICE)
results['Full_MRPC'] = run_training(model_full_mrpc, mrpc_train, mrpc_val, task='accuracy', epochs=3, lr=2e-5)

# ---- CoLA full fine-tune ----
print('\n=== Full Fine-Tune: CoLA ===')
model_full_cola = RobertaClassifier(num_labels=2).to(DEVICE)
results['Full_CoLA'] = run_training(model_full_cola, cola_train, cola_val, task='matthews', epochs=3, lr=2e-5)

# ---- STS-B full fine-tune ----
print('\n=== Full Fine-Tune: STS-B ===')
model_full_stsb = RobertaClassifier(num_labels=1).to(DEVICE)
results['Full_STSB'] = run_training(model_full_stsb, stsb_train, stsb_val, task='regression', epochs=3, lr=2e-5)

## 6. LoRA Fine-Tuning

In [ ]:
LORA_R     = 8
LORA_ALPHA = 16

# ---- MRPC LoRA ----
print('=== LoRA Fine-Tune: MRPC ===')
model_lora_mrpc = RobertaClassifier(num_labels=2).to(DEVICE)
# Freeze all base parameters first
for p in model_lora_mrpc.roberta.parameters():
    p.requires_grad_(False)
apply_lora_to_roberta(model_lora_mrpc, r=LORA_R, alpha=LORA_ALPHA)
results['LoRA_MRPC'] = run_training(model_lora_mrpc, mrpc_train, mrpc_val, task='accuracy', epochs=3, lr=2e-4)

# ---- CoLA LoRA ----
print('\n=== LoRA Fine-Tune: CoLA ===')
model_lora_cola = RobertaClassifier(num_labels=2).to(DEVICE)
for p in model_lora_cola.roberta.parameters():
    p.requires_grad_(False)
apply_lora_to_roberta(model_lora_cola, r=LORA_R, alpha=LORA_ALPHA)
results['LoRA_CoLA'] = run_training(model_lora_cola, cola_train, cola_val, task='matthews', epochs=3, lr=2e-4)

# ---- STS-B LoRA ----
print('\n=== LoRA Fine-Tune: STS-B ===')
model_lora_stsb = RobertaClassifier(num_labels=1).to(DEVICE)
for p in model_lora_stsb.roberta.parameters():
    p.requires_grad_(False)
apply_lora_to_roberta(model_lora_stsb, r=LORA_R, alpha=LORA_ALPHA)
results['LoRA_STSB'] = run_training(model_lora_stsb, stsb_train, stsb_val, task='regression', epochs=3, lr=2e-4)

## 7. Results Comparison Table

In [ ]:
print('\n' + '='*60)
print(f'{"Method":<20} {"MRPC (Acc)":>12} {"CoLA (MCC)":>12} {"STS-B (Pear)":>14}')
print('-'*60)
print(f'{"Full Fine-Tune":<20} {results["Full_MRPC"]:>12.4f} {results["Full_CoLA"]:>12.4f} {results["Full_STSB"]:>14.4f}')
print(f'{"LoRA (r=8)":<20} {results["LoRA_MRPC"]:>12.4f} {results["LoRA_CoLA"]:>12.4f} {results["LoRA_STSB"]:>14.4f}')
print('='*60)
print('\nPaper targets (Table 2, RoBERTa-base):')
print(f'  Full fine-tune: MRPC ~90.9, CoLA ~63.6, STS-B ~91.2')
print(f'  LoRA:           MRPC ~89.7, CoLA ~63.4, STS-B ~91.5')

## 8. Trainable Parameter Count

In [ ]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total_full, train_full = count_params(model_full_mrpc)
total_lora, train_lora = count_params(model_lora_mrpc)

print(f'Full fine-tune : {train_full:,} / {total_full:,} trainable ({100*train_full/total_full:.1f}%)')
print(f'LoRA (r={LORA_R})    : {train_lora:,} / {total_lora:,} trainable ({100*train_lora/total_lora:.2f}%)')

## 9. LoRA Inference Example
At inference time, the LoRA layers are already part of the forward pass — no weight merging needed.

In [ ]:
model_lora_mrpc.eval()

sentence1 = "The cat sat on the mat."
sentence2 = "A cat was sitting on a mat."

enc = tokenizer(
    sentence1, sentence2,
    return_tensors='pt', truncation=True, padding='max_length', max_length=MAX_LEN
)

with torch.no_grad():
    logits = model_lora_mrpc(
        enc['input_ids'].to(DEVICE),
        enc['attention_mask'].to(DEVICE)
    )

pred = logits.argmax(dim=-1).item()
label_map = {0: 'Not equivalent', 1: 'Equivalent'}
print(f'Sentence 1: {sentence1}')
print(f'Sentence 2: {sentence2}')
print(f'Prediction: {label_map[pred]} (logits: {logits.squeeze().tolist()})')